# Extração — Camada RAW (MinIO)

Este notebook faz a extração dos arquivos brutos da pasta `dados_brutos/` (CSV e PDF) e os envia **sem nenhuma transformação** para o MinIO, na camada **Raw**.

A estrutura de pastas original é preservada dentro do bucket, sob o prefixo `dados_brutos/`.

**Por que uma camada Raw à parte, byte-idêntica ao arquivo original:** é o princípio básico da arquitetura medalhão (Raw -> Bronze -> Silver -> ...). Guardar o dado exatamente como veio da fonte, sem nenhum tratamento, garante que qualquer etapa seguinte (extração, limpeza, tradução) possa ser **reexecutada do zero** se um bug for encontrado no parser, sem depender do arquivo original ainda estar disponível no computador local. A Raw é a fonte da verdade; tudo o que vem depois é reproduzível a partir dela.

## 1. Imports

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from minio import Minio
from minio.error import S3Error

## 2. Configuração

Lê o `.env` na raiz do projeto para obter o caminho local dos arquivos brutos (`RAW_DATA_DIR`) e os dados de conexão com o MinIO. `BUCKET_RAW` tem um valor padrão (`"raw"`) caso a variável não esteja definida, para o notebook não quebrar por falta de configuração opcional.

O ajuste de `RAW_DATA_DIR` logo abaixo (checando se a pasta existe relativa ao notebook ou um nível acima) existe porque o mesmo notebook pode ser executado tanto a partir da raiz do projeto quanto de dentro de `notebooks/`, dependendo de como o Jupyter foi iniciado — sem essa checagem, o caminho relativo quebraria conforme o diretório de trabalho.

In [ ]:
# Carrega variáveis do .env (na raiz do projeto)
load_dotenv(Path.cwd().parent / ".env")

RAW_DATA_DIR = Path(os.getenv("RAW_DATA_DIR", "./dados_brutos")).resolve()
if not (Path.cwd() / RAW_DATA_DIR).exists() and (Path.cwd().parent / "dados_brutos").exists():
    RAW_DATA_DIR = (Path.cwd().parent / "dados_brutos").resolve()

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
BUCKET_RAW = os.getenv("BUCKET_RAW", "raw")

print(f"RAW_DATA_DIR : {RAW_DATA_DIR}")
print(f"MINIO_ENDPOINT: {MINIO_ENDPOINT}")
print(f"BUCKET_RAW : {BUCKET_RAW}")

RAW_DATA_DIR : C:\Projeto_AI\dados_brutos
MINIO_ENDPOINT: localhost:9000
BUCKET_RAW : raw


## 3. Conexão com o MinIO e criação do bucket

Cria o bucket `raw` se ele ainda não existir (`bucket_exists`/`make_bucket`). Essa checagem torna o notebook **idempotente**: pode ser executado várias vezes (por exemplo, após adicionar um novo PDF a `dados_brutos/`) sem falhar por já existir, e sem exigir que alguém crie o bucket manualmente antes da primeira execução.

In [4]:
client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False,
)

if not client.bucket_exists(BUCKET_RAW):
    client.make_bucket(BUCKET_RAW)
    print(f"Bucket '{BUCKET_RAW}' criado.")
else:
    print(f"Bucket '{BUCKET_RAW}' já existe.")

Bucket 'raw' criado.


## 4. Listagem dos arquivos brutos

Lista todo arquivo dentro de `dados_brutos/` (via `rglob("*")`, que percorre subpastas — por isso `csv/DATASUS/`, `csv/IBGE/` e `pdf/` são todos encontrados) antes de enviar qualquer coisa. Listar e imprimir primeiro serve como conferência manual: dá pra checar visualmente se a contagem e os nomes batem com o que se espera, antes de gastar tempo/banda subindo tudo para o MinIO.

In [5]:
def listar_arquivos_brutos(raw_dir: Path):
    """Lista todos os arquivos dentro de raw_dir, ignorando diretórios."""
    return [p for p in raw_dir.rglob("*") if p.is_file()]


arquivos = listar_arquivos_brutos(RAW_DATA_DIR)
print(f"{len(arquivos)} arquivo(s) encontrado(s) em {RAW_DATA_DIR}")
for a in arquivos:
    print(" -", a.relative_to(RAW_DATA_DIR))

16 arquivo(s) encontrado(s) em C:\Projeto_AI\dados_brutos
 - csv\DATASUS\AIH-aprovadas_Procedimentos-hospitalares.csv
 - csv\DATASUS\Qtd.aprovada_Producao-Ambulatorial.csv
 - csv\DATASUS\Valor-aprovado_Producao-Ambulatorial.csv
 - csv\DATASUS\Valor-total_Procedimentos-hospitalares.csv
 - csv\IBGE\escolaridade.csv
 - csv\IBGE\populacao.csv
 - csv\IBGE\renda.csv
 - pdf\global-survey-full-report-2019-english.pdf
 - pdf\globalstatistics2016-1.pdf
 - pdf\isaps-global-survey-2024.pdf
 - pdf\isaps-global-survey-results-2018-1.pdf
 - pdf\isaps-global-survey_2020.pdf
 - pdf\isaps-global-survey_2021.pdf
 - pdf\isaps-global-survey_2022.pdf
 - pdf\isaps-global-survey_2023.pdf
 - pdf\isaps_2017_international_study_cosmetic_procedures_new.pdf


## 5. Upload para a camada Raw

Envia cada arquivo para o bucket `raw`, **preservando a estrutura de pastas original** sob o prefixo `dados_brutos/` (ex.: `dados_brutos/pdf/isaps-global-survey_2024.pdf`). Manter o caminho original como nome do objeto é o que permite à camada Bronze (ver [extracao_bronze_isaps.ipynb](extracao_bronze_isaps.ipynb)) filtrar por prefixo (`dados_brutos/pdf/`) e saber exatamente de qual arquivo cada linha extraída veio — sem isso, a rastreabilidade da linha até o PDF de origem se perderia.

In [6]:
def upload_para_raw(client: Minio, bucket: str, raw_dir: Path, arquivos: list[Path]):
    """Envia cada arquivo para o bucket raw, preservando a estrutura de pastas
    original sob o prefixo 'dados_brutos/'."""
    for arquivo in arquivos:
        caminho_relativo = arquivo.relative_to(raw_dir)
        object_name = f"dados_brutos/{caminho_relativo.as_posix()}"

        client.fput_object(
            bucket_name=bucket,
            object_name=object_name,
            file_path=str(arquivo),
        )
        print(f"Enviado: {arquivo.name} -> s3://{bucket}/{object_name}")


upload_para_raw(client, BUCKET_RAW, RAW_DATA_DIR, arquivos)

Enviado: AIH-aprovadas_Procedimentos-hospitalares.csv -> s3://raw/dados_brutos/csv/DATASUS/AIH-aprovadas_Procedimentos-hospitalares.csv
Enviado: Qtd.aprovada_Producao-Ambulatorial.csv -> s3://raw/dados_brutos/csv/DATASUS/Qtd.aprovada_Producao-Ambulatorial.csv
Enviado: Valor-aprovado_Producao-Ambulatorial.csv -> s3://raw/dados_brutos/csv/DATASUS/Valor-aprovado_Producao-Ambulatorial.csv
Enviado: Valor-total_Procedimentos-hospitalares.csv -> s3://raw/dados_brutos/csv/DATASUS/Valor-total_Procedimentos-hospitalares.csv
Enviado: escolaridade.csv -> s3://raw/dados_brutos/csv/IBGE/escolaridade.csv
Enviado: populacao.csv -> s3://raw/dados_brutos/csv/IBGE/populacao.csv
Enviado: renda.csv -> s3://raw/dados_brutos/csv/IBGE/renda.csv
Enviado: global-survey-full-report-2019-english.pdf -> s3://raw/dados_brutos/pdf/global-survey-full-report-2019-english.pdf
Enviado: globalstatistics2016-1.pdf -> s3://raw/dados_brutos/pdf/globalstatistics2016-1.pdf
Enviado: isaps-global-survey-2024.pdf -> s3://raw/dad

## 6. Conferência

Relista os objetos gravados direto do MinIO, em vez de simplesmente confiar que o loop de upload não lançou nenhum erro. É a mesma lógica de qualquer etapa de carga de dados: a confirmação de que o dado está no destino deve vir de uma leitura do próprio destino, não da ausência de exceção no código que escreveu.

In [7]:
# Confirmação: lista o que ficou gravado na camada raw
print(f"Objetos em s3://{BUCKET_RAW}/dados_brutos/:\n")
for obj in client.list_objects(BUCKET_RAW, prefix="dados_brutos/", recursive=True):
    print(f" - {obj.object_name} ({obj.size} bytes)")

Objetos em s3://raw/dados_brutos/:

 - dados_brutos/csv/DATASUS/AIH-aprovadas_Procedimentos-hospitalares.csv (230447 bytes)
 - dados_brutos/csv/DATASUS/Qtd.aprovada_Producao-Ambulatorial.csv (368993 bytes)
 - dados_brutos/csv/DATASUS/Valor-aprovado_Producao-Ambulatorial.csv (402224 bytes)
 - dados_brutos/csv/DATASUS/Valor-total_Procedimentos-hospitalares.csv (369740 bytes)
 - dados_brutos/csv/IBGE/escolaridade.csv (3242 bytes)
 - dados_brutos/csv/IBGE/populacao.csv (2268 bytes)
 - dados_brutos/csv/IBGE/renda.csv (5397 bytes)
 - dados_brutos/pdf/global-survey-full-report-2019-english.pdf (7027261 bytes)
 - dados_brutos/pdf/globalstatistics2016-1.pdf (2356452 bytes)
 - dados_brutos/pdf/isaps-global-survey-2024.pdf (2855390 bytes)
 - dados_brutos/pdf/isaps-global-survey-results-2018-1.pdf (6290587 bytes)
 - dados_brutos/pdf/isaps-global-survey_2020.pdf (4387543 bytes)
 - dados_brutos/pdf/isaps-global-survey_2021.pdf (3579652 bytes)
 - dados_brutos/pdf/isaps-global-survey_2022.pdf (4504223